## Make animations of SIC, SSS, and SST

In [1]:
## import required packages
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
from pyproj import Proj, Transformer
import glob
import os
import string
from datetime import datetime

In [2]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:43459")
client

<Client: 'tcp://127.0.0.1:43459' processes=8 threads=32, memory=123.95 GiB>

#### Create custom colormap for sea ice

In [3]:
def remap_colormap(cmap, nonlin_scale=np.sqrt, n=256):
    """
    Remap a colormap so that lower values are compressed (darker colors take up less space),
    and higher values (lighter colors) are stretched out.
    
    Parameters:
    - cmap: The original colormap (e.g., cmocean.cm.ice)
    - nonlin_scale: A function that maps [0,1] -> [0,1] nonlinearly (default: np.sqrt)
    - n: Number of color levels
    
    Returns:
    - A new colormap with redistributed colors.
    """
    # Generate linearly spaced values, then apply non-linear scaling
    orig_vals = np.linspace(0, 1, n)
    remapped_vals = nonlin_scale(orig_vals)

    # Normalize remapped values to stay in [0,1]
    remapped_vals = (remapped_vals - remapped_vals.min()) / (remapped_vals.max() - remapped_vals.min())

    # Map those values through the original colormap
    new_colors = cmap(remapped_vals)
    
    return LinearSegmentedColormap.from_list(f'remapped_{cmap.name}', new_colors, N=n)

In [4]:
from matplotlib.colors import LinearSegmentedColormap

# create a nonlinearly adjusted colormap from the cmocean ice colormap
custom_ice_cmap = remap_colormap(cmocean.cm.ice, nonlin_scale=np.sqrt)  # sqrt compresses low end

## Plot

In [5]:
def convert_to_doy_strings(dates):
    """
    Convert a list of date strings (YYYYMMDD) to 'YYYY_DDD' format where DDD is day of year.
    """
    doy_strings = []
    for date_str in dates:
        dt = datetime.strptime(date_str, '%Y%m%d')
        doy = dt.timetuple().tm_yday
        doy_strings.append(f"{dt.year}_{doy:03d}")
    return doy_strings

In [31]:
def plot_satellite_snapshot_row(sic_file_path, sss_file_path, sst_file_path, save_folder, i):
    plt.rcParams['font.size'] = 11

    # Projection setup
    nsidc_proj = Proj(proj='stere', lat_0=90, lon_0=-45, lat_ts=70, a=6378273, b=6356889.449, units='m')
    transformer = Transformer.from_proj(nsidc_proj, "epsg:4326", always_xy=True)

    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3),
                             subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-155)})
    plt.subplots_adjust(wspace=0.1, right=0.9, top=0.88)

    # Load SIC (always required)
    sic_cb = None
    if sic_file_path is not None:
        sic_ds = xr.open_dataset(sic_file_path)
        sic = sic_ds['cdr_seaice_conc'].squeeze()
        if sic.max() > 1.5:
            sic = sic / 100.0
        sic_masked = np.where(sic > 0.15, sic, np.nan)
        x = sic_ds['x'].values
        y = sic_ds['y'].values
        X, Y = np.meshgrid(x, y)
        lon, lat = transformer.transform(X, Y)
        dt = pd.to_datetime(sic_ds['time'].values)
    else:
        raise ValueError("SIC file is required to define coordinates and date.")

    # Load SSS if available
    sss_cb = None
    if sss_file_path is not None:
        try:
            sss = xr.open_dataset(sss_file_path)['sss_smap'].squeeze()
        except Exception as e:
            print(f"Failed to load SSS from {sss_file_path}: {e}")
            sss = None

    # Load SST if available
    sst_cb = None
    if sst_file_path is not None:
        try:
            sst_ds = xr.open_dataset(sst_file_path)
            sst = sst_ds['sst'].squeeze()
            sst_ice = sst_ds['ice'].squeeze()
        except Exception as e:
            print(f"Failed to load SST from {sst_file_path}: {e}")
            sst = None
            sst_ice = None

    # Add date title
    date_str = dt.strftime('%b %d, %Y')
    fig.text(0.5, 0.96, date_str[0], ha='center', fontsize=15)

    for j, ax in enumerate(axes):
        ax.set_extent([-161, -126, 69, 78], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m', facecolor='lightgray'))
        ax.add_feature(cfeature.BORDERS.with_scale('10m'))
        ax.add_feature(cfeature.COASTLINE.with_scale('10m'))
        ax.gridlines(draw_labels=False, linewidth=0.5, color='gray',
                     alpha=0.5, linestyle='--', ylocs=[67, 69, 71, 73, 75, 77, 79])

        # Plot SSS on left, SST on right (if available)
        if j == 0 and sss_file_path is not None and sss is not None:
            sss_cb = sss.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.haline,
                              vmin=22, vmax=31, add_colorbar=False, extend='both')
            ax.set_title("SSS and SIC" if sss is not None else "")
        elif j == 1 and sst_file_path is not None and sst is not None and sst_ice is not None:
            sst_cb = sst.where(sst_ice.isnull()).plot(ax=ax, transform=ccrs.PlateCarree(),
                                                      cmap=cmocean.cm.thermal, vmin=-2, vmax=5,
                                                      add_colorbar=False, extend='both')
            ax.set_title("SST and SIC" if sst is not None else "")

        # Plot SIC overlay on both panels if available
        if sic_file_path is not None:
            sic_cb = ax.pcolormesh(lon, lat, sic_masked * 100, transform=ccrs.PlateCarree(),
                                   cmap=custom_ice_cmap, shading='auto', vmin=0, vmax=100)
            regional_mask = ((lon >= -165) & (lon <= -120) & (lat >= 66) & (lat <= 80))
            sic_region = np.where(regional_mask, sic, np.nan)
            ax.contour(lon, lat, sic_region * 100, levels=[15], colors='magenta',
                       linewidths=1, transform=ccrs.PlateCarree())

    # Colorbars only for what's plotted
    if sss_cb:
        cb_sss = fig.colorbar(sss_cb, cax=fig.add_axes([0.13, 0.02, 0.36, 0.05]),
                              orientation='horizontal', extend='both')
        cb_sss.set_label('SSS')
        cb_sss.set_ticks([23, 25, 27, 29, 31])

    if sst_cb:
        cb_sst = fig.colorbar(sst_cb, cax=fig.add_axes([0.535, 0.02, 0.36, 0.05]),
                              orientation='horizontal', extend='both')
        cb_sst.set_label('SST (°C)')
        cb_sst.set_ticks([-2,0,2,4])

    if sic_cb:
        cb_sic = fig.colorbar(sic_cb, cax=fig.add_axes([0.13, -0.2, 0.36, 0.05]),
                              orientation='horizontal')
        cb_sic.set_label('SIC (%)')
        cb_sic.ax.vlines(15, *cb_sic.ax.get_xlim(), colors='magenta', linewidth=1.5)

    # Save and close
    filename = f"satellite_snaps_{i:04d}.png"
    save_path = os.path.join(save_folder, filename)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    # return fig

In [20]:
year = 2022

In [21]:
# Set data directory and load all matching NetCDF files
sic_dir = "/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sic_nsidc_cdr"
sic_file_paths_all = sorted(glob.glob(os.path.join(sic_dir, f"*{year}*.nc")))

sss_dir = "/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sss_rss_smap"
sss_file_paths_all = sorted(glob.glob(os.path.join(sss_dir, f"*{year}*.nc")))

sst_dir = "/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sst_noaa_oisst"
sst_file_paths_all = sorted(glob.glob(os.path.join(sst_dir, f"*{year}*.nc")))

In [22]:
# convert sss from doy to time
doy_dates = convert_to_doy_strings(['20150801'])
print(doy_dates)

sss_file_paths = [item for item in sss_file_paths_all if doy_dates[0] in item]

['2015_213']


In [23]:
print(sic_file_paths_all[0])
print(sss_file_paths_all[0])
print(sst_file_paths_all[0])

/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sic_nsidc_cdr/sic_psn25_20120220_F17_v05r00.nc
/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sss_rss_smap/RSS_smap_SSS_L3_8day_running_2022_001_FNL_v06.0.nc
/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sst_noaa_oisst/oisst-avhrr-v02r01.20220801.nc


In [32]:
fig = plot_satellite_snapshot_row(
    sic_file_path=sic_file_paths_all[0],
    sss_file_path=sss_file_paths_all[0],
    sst_file_path=sst_file_paths_all[0],
    save_folder='/home/jpluser/efs-mount-point/mzahn/animations/png_tmp',i=0)

## Generate pngs for each day

In [33]:
def batch_plot_satellite_snapshots_by_year(year, sic_dir, sss_dir, sst_dir, save_folder):
    # Make sure output folder exists
    os.makedirs(save_folder, exist_ok=True)

    # Gather files for the year
    sic_files = sorted(glob.glob(os.path.join(sic_dir, f"*{year}*.nc")))
    sss_files = sorted(glob.glob(os.path.join(sss_dir, f"*{year}*.nc")))
    sst_files = sorted(glob.glob(os.path.join(sst_dir, f"*{year}*.nc")))

    print(f"Found {len(sic_files)} SIC, {len(sss_files)} SSS, and {len(sst_files)} SST files for {year}.")

    # Build date-to-file dictionaries
    def build_date_dict(file_list, time_var='time'):
        date_dict = {}
        for f in file_list:
            try:
                with xr.open_dataset(f) as ds:
                    date = pd.to_datetime(ds[time_var].values.item()).date()
                    if date.month >= 8:  # Keep only August–December
                        date_dict[date] = f
            except Exception as e:
                print(f"Error reading {f}: {e}")
        return date_dict

    sic_dict = build_date_dict(sic_files)
    sss_dict = build_date_dict(sss_files)
    sst_dict = build_date_dict(sst_files)

    # Combine all unique dates from any dataset
    all_dates = sorted(set(sic_dict) | set(sss_dict) | set(sst_dict))

    print(f"Generating plots for {len(all_dates)} dates (Aug–Dec {year}).")

    for i, date in enumerate(all_dates):
        sic_fp = sic_dict.get(date)
        sss_fp = sss_dict.get(date)
        sst_fp = sst_dict.get(date)

        plot_satellite_snapshot_row(sic_fp, sss_fp, sst_fp, save_folder, i)

    print(f"Finished saving {len(all_dates)} images to {save_folder}.")

In [34]:
sic_dict = batch_plot_satellite_snapshots_by_year(
    year=2022,
    sic_dir="/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sic_nsidc_cdr",
    sss_dir="/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sss_rss_smap",
    sst_dir="/home/jpluser/efs-mount-point/mzahn/data/satellite_data/sst_noaa_oisst",
    save_folder="/home/jpluser/efs-mount-point/mzahn/animations/satellite_snap_figures/2022"
)

Found 163 SIC, 317 SSS, and 153 SST files for 2022.
Generating plots for 153 dates (Aug–Dec 2022).
Finished saving 153 images to /home/jpluser/efs-mount-point/mzahn/animations/satellite_snap_figures/2022.


***

Then run in the command line:<br>
`ffmpeg -r 5 -i satellite_snaps_%04d.png -pix_fmt yuv420p -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" -crf 10 sassie-beaufort-satellite-2022.mp4`<br>

`aws s3 cp ~/efs-mount-point/mzahn/animations/satellite_snap_figures/2022/sassie-beaufort-satellite-2022.mp4 s3://ecco-processed-data/SASSIE/Videos/`